# ViSoBERT -- Combined augmentation (BT + EDA + LLM + VOZ)

Train trên subset `combined_segmented` (đã phân từ sẵn trên Hugging Face Hub, đã gộp baseline + BT + EDA + LLM + VOZ và loại conflict/leakage), đánh giá trên `tdtu_dev` / `tdtu_test` (đã đóng băng).

**Khác biệt so với bản PhoBERT:**
- Model: `uitnlp/visobert` (XLM-R based, train trên Vietnamese social media text).
- Tokenizer: dùng **fast** tokenizer (`use_fast=True`).
- **Không** áp dụng word segmentation -- bỏ dấu `_` khỏi cột `text`.
- Bỏ dependency `py_vncorenlp`.

**Chạy trên Google Colab** -- cần mount Drive và có sẵn secret `HF_TOKEN`.

## Dependencies

In [ ]:
!pip install -q transformers datasets huggingface_hub scikit-learn accelerate sentencepiece
!pip install -q -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 23.1 MB/s eta 0:00:00


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: Enable GPU in Runtime > Change runtime type.")

Device: cuda
GPU: Tesla T4


In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami, HfApi

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token, add_to_git_credential=False)
user_info = whoami()
print(f"Authenticated as: {user_info['name']}")

Authenticated as: AnoraLee


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Hate_Speech_Detection")
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_DIR}")

Mounted at /content/drive
Project root: /content/drive/MyDrive/Hate_Speech_Detection


In [ ]:
MODEL_NAME = "uitnlp/visobert"
MAX_LENGTH = 128
SEED = 42
LABELS = ["CLEAN", "OFFENSIVE", "HATE"]

label2id = {label: index for index, label in enumerate(LABELS)}
id2label = {index: label for label, index in label2id.items()}

# Repo data trên Hugging Face Hub
HF_DATA_REPO = "AnoraLee/tdtu_vietnamese_hsd_final"
SUBSET_NAME = "combined_segmented"

## Load dữ liệu từ Hugging Face Hub + chuẩn hoá cho ViSoBERT

Data trên Hub đã được gộp 5 nguồn + loại conflict/leakage + phân từ sẵn. Vì ViSoBERT không dùng phân từ, ta **bỏ dấu `_`** trong cột `text` để quay về raw. Nếu Hub có sẵn `text_raw`, sẽ ưu tiên dùng nó.

In [ ]:
from datasets import load_dataset
import pandas as pd

REPO_DATA = "AnoraLee/tdtu_vietnamese_hsd_final"

print("Đang tải tập Train [combined_segmented] từ Hugging Face...")
dataset_train = load_dataset(REPO_DATA, "combined_segmented")
train_df = dataset_train["tdtu_train"].to_pandas()

print("Đang tải tập Dev/Test [baseline_segmented] từ Hugging Face...")
dataset_eval = load_dataset(REPO_DATA, "baseline_segmented")
dev_df = dataset_eval["tdtu_dev"].to_pandas()
test_df = dataset_eval["tdtu_test"].to_pandas()

def ensure_raw_text(df):
    df = df.copy()
    if "text_raw" not in df.columns:
        df["text_raw"] = df["text"].astype(str).str.replace("_", " ")
        df["text_raw"] = df["text_raw"].str.replace(r"\s+", " ", regex=True).str.strip()
    return df

train_df = ensure_raw_text(train_df)
dev_df = ensure_raw_text(dev_df)
test_df = ensure_raw_text(test_df)

train_df["text"] = train_df["text_raw"]
dev_df["text"] = dev_df["text_raw"]
test_df["text"] = test_df["text_raw"]

train_df = train_df.dropna(subset=["text", "label"]).reset_index(drop=True)
dev_df = dev_df.dropna(subset=["text", "label"]).reset_index(drop=True)
test_df = test_df.dropna(subset=["text", "label"]).reset_index(drop=True)

print(f"Kích thước Train: {train_df.shape}")
print(f"Kích thước Dev: {dev_df.shape}")
print(f"Kích thước Test: {test_df.shape}")

Đang tải tập Train [combined_segmented] từ Hugging Face...
Đang tải tập Dev/Test [baseline_segmented] từ Hugging Face...
Kích thước Train: (140914, 4)
Kích thước Dev: (2650, 3)
Kích thước Test: (6576, 3)


## Tokenizer + hàm dùng chung

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

def prepare_split(df):
    output = df[["text", "label"]].dropna().copy()
    assert output["label"].isin(LABELS).all(), "Unexpected label found."
    output["label"] = output["label"].map(label2id)
    return Dataset.from_pandas(output, preserve_index=False)

def build_tokenized_dataset(train_df, dev_df, test_df):
    dataset = DatasetDict({
        "train": prepare_split(train_df),
        "validation": prepare_split(dev_df),
        "test": prepare_split(test_df),
    })
    tokenized = dataset.map(tokenize_function, batched=True)
    tokenized = tokenized.rename_column("label", "labels")
    model_columns = [
        c for c in ["input_ids", "attention_mask", "labels"]
        if c in tokenized["train"].column_names
    ]
    tokenized.set_format("torch", columns=model_columns)
    print(tokenized)
    return tokenized

config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B /  471kB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

In [ ]:
tokenized = build_tokenized_dataset(train_df, dev_df, test_df)

Map:   0%|          | 0/140914 [00:00<?, ? examples/s]

Map:   0%|          | 0/2650 [00:00<?, ? examples/s]

Map:   0%|          | 0/6576 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 140914
    })
    validation: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 2650
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 6576
    })
})


## Metrics

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    """Unified metric schema -- MUST stay identical across all notebooks
    so results in the report are directly comparable."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "weighted_f1": f1_score(labels, predictions, average="weighted"),
        "hate_f1": f1_score(labels, predictions, labels=[LABELS.index("HATE")], average="macro"),
    }

## Train

In [ ]:
import json
from transformers import TrainingArguments, AutoModelForSequenceClassification, Trainer, set_seed, EarlyStoppingCallback

set_seed(SEED)

def train_experiment(experiment_name, model_dir_name, tokenized, push_to_hub_repo=None):
    """Load a FRESH ViSoBERT model and train one experiment end to end.
    Always instantiates a new model -- never reuses a model/trainer object
    from a previous cell, to avoid accidental weight leakage between runs."""
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, label2id=label2id, id2label=id2label,
    )
    model_dir = MODELS_DIR / model_dir_name

    training_args = TrainingArguments(
        output_dir=str(model_dir / "_checkpoints"),
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=10,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        report_to="none",
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()

    test_metrics = trainer.evaluate(tokenized["test"], metric_key_prefix="test")
    print(f"[{experiment_name}] test metrics: {test_metrics}")

    trainer.save_model(model_dir)
    tokenizer.save_pretrained(model_dir)

    metrics_dir = RESULTS_DIR / "metrics"
    metrics_dir.mkdir(parents=True, exist_ok=True)

    metrics_path = metrics_dir / f"metrics_{experiment_name}.json"
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(test_metrics, f, indent=2, ensure_ascii=False)

    print(f"Model saved to:   {model_dir}")
    print(f"Metrics saved to: {metrics_path}")

    if push_to_hub_repo:
        api = HfApi()
        api.create_repo(repo_id=push_to_hub_repo, repo_type="model", private=True, exist_ok=True)
        trainer.model.push_to_hub(push_to_hub_repo, private=True,
                                  commit_message=f"Upload ViSoBERT {experiment_name} model")
        tokenizer.push_to_hub(push_to_hub_repo, private=True,
                              commit_message="Upload ViSoBERT tokenizer")
        api.upload_file(
            path_or_fileobj=str(metrics_path),
            path_in_repo="test_metrics.json",
            repo_id=push_to_hub_repo, repo_type="model",
        )
        print(f"Uploaded: https://huggingface.co/{push_to_hub_repo}")

    return trainer, test_metrics

In [ ]:
trainer, test_metrics = train_experiment(
    experiment_name="visobert_combined",
    model_dir_name="combined_visobert",
    tokenized=tokenized,
    push_to_hub_repo=f"{whoami()['name']}/vietnamese-hsd-visobert-combined",
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  390MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: uitnlp/visobert
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors: reconstructing file:   0%|          |  0.00B /  390MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...iyub3q2/model.safetensors:   0%|          |  553kB /  390MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Uploaded: https://huggingface.co/AnoraLee/vietnamese-hsd-visobert-combined
